In [2]:
import duckdb
import pandas as pd
import os
from datetime import datetime

In [3]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)


In [4]:
arquivo = 'z0019_2.csv'
data_ingestao = datetime.now()
df = pd.read_csv(f'../landing/{arquivo}', sep=';')
df['nome_arquivo'] = arquivo
df['data_ingestao'] = data_ingestao
df.head()

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10004,SERRA,BT50,100,200,z0019_2.csv,2025-12-27 13:04:25.011682
1,10005,MACHADO,BT50,100,100,z0019_2.csv,2025-12-27 13:04:25.011682
2,10003,PREGO,BT10,100,60,z0019_2.csv,2025-12-27 13:04:25.011682


In [5]:
con.execute("""
    CREATE TABLE IF NOT EXISTS bronze_produtos(
            NATBR VARCHAR,
            MAKTX VARCHAR,
            WERKS VARCHAR,
            MAINS VARCHAR,
            LABST VARCHAR,
            nome_arquivo VARCHAR,
            data_ingestao TIMESTAMP
        )
""")

In [6]:
con.execute("INSERT INTO bronze_produtos SELECT * FROM df")

In [7]:
resultado = con.execute("SELECT * FROM bronze_produtos").fetchdf()
print (resultado)

   NATBR     MAKTX WERKS MAINS LABST nome_arquivo              data_ingestao
0  10003     PREGO  BT10   100    60  z0019_2.csv 2025-12-26 18:14:18.326029
1  10002   MARTELO  BT50   100  1500  z0019_1.csv 2025-12-26 18:04:06.139476
2  10001  PARAFUSO  BT10   100   100  z0019_1.csv 2025-12-26 18:04:06.139476
3  10005   MACHADO  BT50   100   100  z0019_2.csv 2025-12-26 18:14:18.326029
4  10003     PREGO  BT10   100    50  z0019_1.csv 2025-12-26 18:04:06.139476
5  10004     SERRA  BT50   100   200  z0019_2.csv 2025-12-26 18:14:18.326029
6  10004     SERRA  BT50   100   200  z0019_2.csv 2025-12-27 13:04:25.011682
7  10005   MACHADO  BT50   100   100  z0019_2.csv 2025-12-27 13:04:25.011682
8  10003     PREGO  BT10   100    60  z0019_2.csv 2025-12-27 13:04:25.011682


In [8]:
# O comando "CREATE OR REPLACE" substitui a tabela antiga pela nova consulta
con.execute("""
    CREATE OR REPLACE TABLE bronze_produtos AS 
    SELECT DISTINCT * FROM bronze_produtos
""")

# Verificar o resultado
print(con.execute("SELECT * FROM bronze_produtos").fetchdf())

   NATBR     MAKTX WERKS MAINS LABST nome_arquivo              data_ingestao
0  10003     PREGO  BT10   100    60  z0019_2.csv 2025-12-26 18:14:18.326029
1  10005   MACHADO  BT50   100   100  z0019_2.csv 2025-12-27 13:04:25.011682
2  10004     SERRA  BT50   100   200  z0019_2.csv 2025-12-26 18:14:18.326029
3  10005   MACHADO  BT50   100   100  z0019_2.csv 2025-12-26 18:14:18.326029
4  10003     PREGO  BT10   100    50  z0019_1.csv 2025-12-26 18:04:06.139476
5  10004     SERRA  BT50   100   200  z0019_2.csv 2025-12-27 13:04:25.011682
6  10002   MARTELO  BT50   100  1500  z0019_1.csv 2025-12-26 18:04:06.139476
7  10003     PREGO  BT10   100    60  z0019_2.csv 2025-12-27 13:04:25.011682
8  10001  PARAFUSO  BT10   100   100  z0019_1.csv 2025-12-26 18:04:06.139476


In [9]:
con.close()